# ORBIT Gate 3 — Matched Representation Selection & Promotion

**Specification:** `ORBIT_IMPLEMENTATION_BRIEF_v1.3.md` (Architecture Decision DEC-02)  
**Objective:** Evaluate strictly two visual representation candidates for the ORBIT world model:
- **Candidate A (Baseline Comparator):** Clean-room, from-scratch ORBIT spatial encoder (Conv2d, 8x8 grid, 192-dim tokens, learned PosEmb).
- **Candidate B (Preferred Candidate):** Frozen Meta AI DINOv2 ViT-S/14 (21.7M params, 14x14 patches, 384-dim) with full-frame nearest-neighbor resize to 224x224 and trainable spatial adapter mapping to the shared 8x8x192 representation.

### Why Run on Google Colab (GPU):
- **Compute Bottleneck:** Candidate B processes 128 images of 224x224 per training step (B=8, T=16). On Mac CPU, each step takes ~1.2s (~20-25 minutes for 1,000 updates + 40 mins for evaluation). On a Colab T4/A100 GPU with CUDA FP16/AMP, batch inference runs in ~40ms per step (~1.5 to 2.5 minutes total, a **30x speedup**).
- **Promotion Pathway:** If Candidate B demonstrates healthy representations (r_eff >= 10.0, D_active >= 120, S_spatial < 0.90) and consistently outperforms static persistence (RelPredErr < 0.95) and shuffled-action controls, Candidate B is eligible for direct promotion to ORBIT's production visual encoder.

## 1. Environment Setup & Dependency Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world 2>/dev/null || (cd /content/learn-a-world && git pull origin main)
%cd /content/learn-a-world

# Install pinned dependencies
!pip install -q -r requirements.txt
!pip install -q torchvision timm

## 2. Hardware Verification & Cryptographic Provenance Audit

In [ ]:
import hashlib
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

# Pre-cache and verify official Meta AI DINOv2 ViT-S/14 weights
dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True)
dinov2_model.eval().to(device)

# Verify parameter count and patch grid contract
total_params = sum(p.numel() for p in dinov2_model.parameters())
print(f'DINOv2 ViT-S/14 Total Parameters: {total_params} (Expected: 21,665,664)')
assert total_params == 21665664, f'Unexpected parameter count: {total_params}'

# Test forward pass
dummy_img = torch.randn(2, 3, 224, 224, device=device)
with torch.no_grad():
    features = dinov2_model.forward_features(dummy_img)
    patch_tokens = features['x_norm_patchtokens']
    print(f'DINOv2 Spatial Patch Tokens Shape: {patch_tokens.shape} (Expected: [2, 256, 384])')
    assert patch_tokens.shape == (2, 256, 384)
    assert features['x_norm_regtokens'].shape[1] == 0, 'Unexpected register tokens detected!'
print('DINOv2 backbone verified and ready.')

## 3. Run Foundation Unit Tests (Gates 0–2 & Baseline Ladder)

In [ ]:
%cd /content/learn-a-world
!pytest tests/test_orbit_gates_0_to_2.py tests/test_gate0_2_independent_verification.py tests/test_true_jepa.py tests/test_baseline_ladder.py -v

## 4. Run Gate 3 Unit & Invariant Tests

In [ ]:
%cd /content/learn-a-world
!pytest tests/test_orbit_gate3_representation.py -v

## 5. Execute Matched Representation Training (Candidate A vs Candidate B)

Both arms train on the exact same replay data for 1,000 updates under identical batching (B=8, T=16) and optimizer settings (AdamW, lr=3e-4).

In [ ]:
%cd /content/learn-a-world
!python -m experiments.run_gate3_representation_ladder \
  --device cuda \
  --updates 1000 \
  --batch-size 8 \
  --sequence-length 16 \
  --eval-seeds 3030 3031 3032 3033 3034 3035 3036 3037 3038 3039 \
  --output-dir /content/drive/MyDrive/learn-a-world/orbit_gate3

## 6. Evaluate Diagnostics, Effective Rank, & Held-Out Predictive Utility

In [ ]:
import json
from pathlib import Path
import pandas as pd

results_file = Path('/content/drive/MyDrive/learn-a-world/orbit_gate3/gate3_representation_metrics.json')
if results_file.exists():
    with open(results_file, 'r') as f:
        metrics = json.load(f)
    
    df = pd.DataFrame([
        {
            'Candidate': 'Candidate A (From Scratch)',
            'Active Dims (>=120)': metrics['candidate_a']['active_dimensions'],
            'Effective Rank (>=10.0)': f"{metrics['candidate_a']['effective_rank']:.2f}",
            'Spatial Diversity (<0.90)': f"{metrics['candidate_a']['spatial_diversity']:.3f}",
            'Online-Target Drift (<=1.0)': f"{metrics['candidate_a']['target_drift']:.3f}",
            'RelPredErr (<0.95)': f"{metrics['candidate_a']['relative_prediction_error']:.3f}",
            'Shuffled Action Sensitivity': metrics['candidate_a']['shuffled_action_sensitive'],
            'Throughput (steps/s)': f"{metrics['candidate_a']['throughput_fps']:.1f}"
        },
        {
            'Candidate': 'Candidate B (Frozen DINOv2 + Adapter)',
            'Active Dims (>=120)': metrics['candidate_b']['active_dimensions'],
            'Effective Rank (>=10.0)': f"{metrics['candidate_b']['effective_rank']:.2f}",
            'Spatial Diversity (<0.90)': f"{metrics['candidate_b']['spatial_diversity']:.3f}",
            'Online-Target Drift (<=1.0)': f"{metrics['candidate_b']['target_drift']:.3f}",
            'RelPredErr (<0.95)': f"{metrics['candidate_b']['relative_prediction_error']:.3f}",
            'Shuffled Action Sensitivity': metrics['candidate_b']['shuffled_action_sensitive'],
            'Throughput (steps/s)': f"{metrics['candidate_b']['throughput_fps']:.1f}"
        }
    ])
    print(df.to_markdown(index=False))
    print(f"\nFormal Promotion Verdict: {metrics.get('verdict', 'PENDING')}")
else:
    print('Results file not found. Ensure training run completes.')